In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

# 1. Load Data
path = r"../by_month_policy.csv"
df = pd.read_csv(path)
df = df[(df['year'] < 2025) | ((df['year'] == 2025) & (df['month'] <= 7))]

# 2. Preprocessing: Create Time Variable and Log(Alpha)
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
min_date = df['date'].min()
df['Time'] = (df['date'].dt.year * 12 + df['date'].dt.month) - (min_date.year * 12 + min_date.month)
df['Log_Alpha'] = np.log(df['mean'] + 1e-6)

# 3. Filter for Growth Phase (Jan 2023 -> Time >= 24)
growth_data = df[df['Time'] >= 24].copy()

# 4. Prepare Groups and Calculate Deltas (Differencing)
policy_group = growth_data[growth_data['has_ai_policy'] == True].sort_values('Time').set_index('Time')['Log_Alpha']
no_policy_group = growth_data[growth_data['has_ai_policy'] == False].sort_values('Time').set_index('Time')['Log_Alpha']

diff_policy = policy_group.diff().dropna()
diff_no_policy = no_policy_group.diff().dropna()

# 5. Mann-Whitney U Test
u_stat, p_val = stats.mannwhitneyu(diff_policy, diff_no_policy, alternative='two-sided')

# 6. Output
print("\n" + "="*57)
print("General Analysis: Mann-Whitney U Test on Monthly Deltas")
print("="*57)
print(f"  - Policy Group Mean Delta:    {diff_policy.mean():.5f}")
print(f"  - No Policy Group Mean Delta: {diff_no_policy.mean():.5f}")
print(f"  - U Statistic:                {u_stat}")
print(f"  - P Value:                    {p_val:.4f} ({'ns' if p_val >= 0.05 else '*'} )")
print("\n  -> Conclusion: No significant difference in growth speed distribution.")
print("="*57)


General Analysis: Mann-Whitney U Test on Monthly Deltas
  - Policy Group Mean Delta:    0.04155
  - No Policy Group Mean Delta: 0.05889
  - U Statistic:                411.0
  - P Value:                    0.7216 (ns)

  -> Conclusion: No significant difference in growth speed distribution.


In [ ]:
# 1. Load Data
path_country = r"../by_month_policy_country.csv"
df_country = pd.read_csv(path_country)
df_country = df_country[(df_country['year'] < 2025) | ((df_country['year'] == 2025) & (df_country['month'] <= 7))]
df_country = df_country.dropna(subset=['country_list'])

# 2. Preprocessing
english_speaking_countries = {
    'AU', 'BM', 'CA', 'FK', 'GI', 'GG', 'GY', 'IE', 'IM', 'JE', 'NZ', 'SG', 'ZA', 'GB', 'US', 'VI',
    'AG', 'AI', 'BS', 'BB', 'VG', 'KY', 'DM', 'GD', 'VC', 'JM', 'MS', 'KN', 'LC', 'TT', 'TC'
}
df_country['is_english'] = df_country['country_list'].isin(english_speaking_countries)
df_country['date'] = pd.to_datetime(df_country[['year', 'month']].assign(day=1))
min_date = df_country['date'].min()
df_country['Time'] = (df_country['date'].dt.year * 12 + df_country['date'].dt.month) - (min_date.year * 12 + min_date.month)
df_country['Log_Alpha'] = np.log(df_country['mean'] + 1e-6)

def run_u_test_on_deltas(data, start_time=24):
    subset = data[data['Time'] >= start_time].copy()
    # Aggregate trends: Index=Time, Columns=HasPolicy
    trends = subset.pivot_table(index='Time', columns='has_ai_policy', values='Log_Alpha', aggfunc='mean')
    if True not in trends.columns or False not in trends.columns: return np.nan, np.nan, np.nan, np.nan
    
    deltas = trends.diff().dropna()
    u_stat, p_val = stats.mannwhitneyu(deltas[True], deltas[False], alternative='two-sided')
    return deltas[True].mean(), deltas[False].mean(), u_stat, p_val

print("\n" + "="*73)
print("Language Group Analysis: Mann-Whitney U Test (Time >= 24)")
print("="*73)

# English Group
eng_mean_p, eng_mean_np, eng_u, eng_p = run_u_test_on_deltas(df_country[df_country['is_english'] == True])
print(f"\n🌍 English-speaking Countries:")
print(f"  - Policy Mean: {eng_mean_p:.5f} | No Policy Mean: {eng_mean_np:.5f}")
print(f"  - U-Stat: {eng_u:.1f} | P-Value: {eng_p:.4f} ({'ns' if eng_p >= 0.05 else '*'}) ")

# Non-English Group
non_mean_p, non_mean_np, non_u, non_p = run_u_test_on_deltas(df_country[df_country['is_english'] == False])
print(f"\n🌐 Non-English-speaking Countries:")
print(f"  - Policy Mean: {non_mean_p:.5f} | No Policy Mean: {non_mean_np:.5f}")
print(f"  - U-Stat: {non_u:.1f} | P-Value: {non_p:.4f} ({'ns' if non_p >= 0.05 else '*'}) ")
print("="*73)


Language Group Analysis: Mann-Whitney U Test (Time >= 24)

🌍 English-speaking Countries:
  - Policy Mean: 0.07695 | No Policy Mean: -0.02000
  - U-Stat: 470.0 | P-Value: 0.4461 (ns)

🌐 Non-English-speaking Countries:
  - Policy Mean: 0.08850 | No Policy Mean: 0.15158
  - U-Stat: 408.0 | P-Value: 0.8520 (ns)


In [ ]:
# 1. Load Data
path_domain = r"../by_month_policy_domain.csv"
df_domain = pd.read_csv(path_domain)
df_domain = df_domain[(df_domain['year'] < 2025) | ((df_domain['year'] == 2025) & (df_domain['month'] <= 7))]

# 2. Preprocessing
df_domain['date'] = pd.to_datetime(df_domain[['year', 'month']].assign(day=1))
min_date = df_domain['date'].min()
df_domain['Time'] = (df_domain['date'].dt.year * 12 + df_domain['date'].dt.month) - (min_date.year * 12 + min_date.month)
df_domain['Log_Alpha'] = np.log(df_domain['mean'] + 1e-6)

domains = ['Health Sciences', 'Life Sciences', 'Physical Sciences', 'Social Sciences']

print("\n" + "="*85)
print("Domain Analysis: Mann-Whitney U Test on Monthly Deltas")
print("="*85)
print(f"{'Domain':<22} | {'Pol Mean':<10} | {'No Pol Mean':<11} | {'U-Stat':<8} | {'P-Value':<10} | {'Sig'}")
print("-" * 85)

for domain in domains:
    # Run U Test for specific domain
    subset = df_domain[df_domain['domain_list'] == domain]
    mean_p, mean_np, u_val, p_val = run_u_test_on_deltas(subset, start_time=24)
    
    if pd.isna(u_val):
        print(f"{domain:<22} | N/A (Insufficient Data)")
    else:
        sig = 'ns' if p_val >= 0.05 else '*'
        print(f"{domain:<22} | {mean_p:<10.4f} | {mean_np:<11.4f} | {u_val:<8.1f} | {p_val:<10.4f} | {sig}")

print("="*85)


Domain Analysis: Mann-Whitney U Test on Monthly Deltas
Domain                 | Pol Mean   | No Pol Mean | U-Stat   | P-Value    | Sig
-------------------------------------------------------------------------------------
Health Sciences        | 0.0305     | 0.0237      | 456.0    | 0.5862     | ns
Life Sciences          | 0.0344     | 0.0365      | 405.0    | 0.8156     | ns
Physical Sciences      | 0.0543     | 0.0559      | 405.0    | 0.8156     | ns
Social Sciences        | 0.0379     | 0.0478      | 417.0    | 0.9628     | ns


In [ ]:
# 1. Load Data
path_oa = r"../by_month_oa_policy.csv"
df_oa = pd.read_csv(path_oa)

# 2. Preprocessing
df_oa['date'] = pd.to_datetime(df_oa[['year', 'month']].assign(day=1))
min_date = df_oa['date'].min()
df_oa['Time'] = (df_oa['date'].dt.year * 12 + df_oa['date'].dt.month) - (min_date.year * 12 + min_date.month)
df_oa['Log_Alpha'] = np.log(df_oa['mean'] + 1e-6)

print("\n" + "="*85)
print("Open Access (OA) Analysis: Mann-Whitney U Test on Monthly Deltas")
print("="*85)

for label, is_oa_val in [('OA Journals', True), ('Non-OA Journals', False)]:
    subset = df_oa[df_oa['is_oa'] == is_oa_val]
    mean_p, mean_np, u_val, p_val = run_u_test_on_deltas(subset, start_time=24)
    
    print(f"\n📂 {label} ({is_oa_val}):")
    if pd.isna(u_val):
        print("  N/A (Insufficient Data)")
    else:
        sig = 'ns' if p_val >= 0.05 else '*'
        print(f"  - Policy Mean: {mean_p:.5f} | No Policy Mean: {mean_np:.5f}")
        print(f"  - U-Stat: {u_val:.1f} | P-Value: {p_val:.4f} ({sig})")

print("="*85)


Open Access (OA) Analysis: Mann-Whitney U Test on Monthly Deltas

📂 OA Journals (True):
  - Policy Mean: 0.04106 | No Policy Mean: 0.04591
  - U-Stat: 403.0 | P-Value: 0.7915 (ns)

📂 Non-OA Journals (False):
  - Policy Mean: 0.04157 | No Policy Mean: 0.03724
  - U-Stat: 410.0 | P-Value: 0.8764 (ns)
